# Demo 3 · Run Kev-4B and Laya locally

**Part 04 · Local decision models** · about 10 minutes

**Part 1 (always runs):** the saved head-to-head runs. Jev, Kev-4B and Laya on the same SST-2,
AG News, BoolQ and long IMDb examples, with accuracy, macro-F1, failures and latency.

**Part 2 (optional, live):** install Kev-4B and Laya in their own environments and run them on
our support tickets. Kev needs `transformers>=5` and Laya needs `transformers<5`, so each runs in
a separate worker process. Laya runs on CPU; Kev-4B needs a GPU or Apple Silicon to be practical.
Downloads are several GB: do this before the session, not on stage.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dmw import bench, metrics, datagen

for ds in bench.datasets():
    meta = bench.load()[ds]
    print(f"{ds:<10} {len(meta['examples']):>3} examples · {meta['dataset']['name']}")

## Part 1 · The saved head-to-head runs

In [ ]:
tables = {ds: bench.table(ds) for ds in bench.datasets()}
pd.concat(tables, names=["dataset"])

In [ ]:
acc = pd.DataFrame({ds: t["accuracy"] for ds, t in tables.items()}).T * 100
ax = acc.plot.bar(figsize=(8, 3.2), color=["#142B3D", "#075E91", "#1A8A9C"], rot=0)
ax.set_ylabel("accuracy, %"); ax.set_ylim(0, 105); ax.legend(frameon=False, ncol=3)
for c in ax.containers: ax.bar_label(c, fmt="%.0f", fontsize=8)
plt.title("Same examples, failures count as wrong"); plt.tight_layout(); plt.show()

### Where the models disagree (AG News)

Agreement between two models says nothing about who is right: read a few rows.
`attach_texts()` downloads the example texts from the pinned dataset revision.

In [ ]:
ag = bench.frame("ag_news")
dis = ag[(ag["kev-4b"] != ag["jev"]) | (ag["laya"] != ag["jev"])]
print(f"{len(dis)} of {len(ag)} examples have at least one disagreement with Jev")
try:
    dis = bench.attach_texts("ag_news", dis)
    cols = ["text", "gold", "jev", "kev-4b", "laya"]
except Exception as e:
    print("Texts not available offline:", type(e).__name__); cols = ["id", "gold", "jev", "kev-4b", "laya"]
dis[cols].head(8)

### Long input: the encoder refuses instead of guessing

IMDb long reviews have 1,000+ words; Laya's limit is 512 tokens. Every request is rejected,
which the benchmark counts as a failure, not as a wrong label.

In [ ]:
im = bench.frame("imdb_long")
print("Laya failures:", im["laya"].isna().sum(), "of", len(im))
print("Example error:", im["laya_error"].dropna().iloc[0])
print("Words per review: median", int(im.words.median()), "| max", im.words.max())
bench.table("imdb_long")

## Part 2 · Live: run the models yourself (optional)

Set `RUN_LOCAL = True`. Setup creates `~/.dmw_venvs/laya` and `~/.dmw_venvs/kev-4b` (or `$DMW_VENVS`)
and installs the pinned packages. In Colab, use a GPU runtime for Kev-4B. Kev on CUDA with fp16
(T4) is not tested upstream; if it fails, keep Laya live and use Part 1 for Kev.

In [ ]:
RUN_LOCAL = True         # set True to install and run the models
MODELS = ["laya", "kev-4b"]

In [ ]:
from dmw import local
if RUN_LOCAL:
    for m in MODELS:
        if not local.is_ready(m):
            local.setup(m)
        print(m, "ready")

In [ ]:
# Shared examples: the 16 support tickets from the fresh test set (4 labels)
tickets = [c for c in datagen.load_test("data/fresh_test.json") if c["group"] == "tickets"]
LABELS = tickets[0]["choices"]
INSTRUCTIONS = "Which team should handle this support ticket?"

results = []
if RUN_LOCAL:
    for m in MODELS:
        with local.LocalModel(m) as lm:
            print(f"{m}: loaded in {lm.load_s} s on {lm.info['device']}")
            for c in tickets:
                out = lm.predict(c["text"], LABELS, INSTRUCTIONS)
                results.append({"model": m, "id": c["id"], "text": c["text"][:60], "gold": c["expected"],
                                "pred": out.get("label"), "p_top": max(out["probabilities"].values()) if out.get("probabilities") else None,
                                "ms": out["latency_ms"], "error": out.get("error")})
    live = pd.DataFrame(results)
    display(live.pivot(index=["id", "text", "gold"], columns="model", values="pred"))
    print(live.groupby("model").apply(lambda g: pd.Series({
        "accuracy": (g.pred == g.gold).mean(), "p50 ms": g.ms.median()})))
else:
    print("RUN_LOCAL is False: Part 1 shows the saved runs.")

In [ ]:
# The long-input test: one IMDb review of 1,000+ words through Laya (needs RUN_LOCAL and internet)
if RUN_LOCAL:
    im_t = bench.attach_texts("imdb_long", bench.frame("imdb_long").head(1))
    with local.LocalModel("laya") as lm:
        print(lm.predict(im_t.text.iloc[0], bench.labels("imdb_long"), "Classify the sentiment of this text."))

## Takeaways

* Short text: the best local models are within a few points of Jev; Laya leads AG News.
* BoolQ and long documents separate them: encoders cannot read long input at all.
* Load time and hardware are part of the cost of going local.